The following are the steps for using AutoML for a regression task:

_Note: Setting the flag for featurization= 'True' generates represents molecules using 5 representation techniques._ 

1. Requires an input pandas dataframe consisting of two columns:<br>
    * SMILES strings<br>
    * target property values<br>


2. Molecules are represented as:<br>
    * coloumb matrix<br>
    * rdkit morgan fingerprints<br>
    * MACCs<br>
    * rdkit hashed topological torsion<br>
    * rdkit molecular descriptors (all)<br>



3. Screens through various sklearn regressor models:<br>
    * [MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html)<br>
    * [GradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)<br>
    * [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)<br>
    * [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html)<br>
    * [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html)<br>
    * [SVR](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html)<br>
    * [ElasticNet](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)<br>
    * [DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)<br>


Yields 'n-best' models, with optimized hyperparamters.

Returns dataframe of error metrics, machine learning model, algorithm, tuned hyperparameter values and featurization technique.

# Load your data 

In [1]:
import pandas as pd
import numpy as np
from chemml.chem import Molecule
from chemml.datasets import load_organic_density

In [2]:
molecules, target, dragon_subset = load_organic_density()
df=pd.concat([molecules, target], axis=1)
df = df.sample(25, random_state=42)
df

,smiles,density_Kg/m3
361,c1scc(n1)c1ncc(s1)c1c(ccc2c1cccc2)c1cncs1,1328.35
73,S1CC(SC(C1)C1CCCC1)c1cocc1c1ccccc1,1143.80
374,N1CNC(NC1)c1sccc1C1(CSCCS1)c1cccs1,1351.33
155,c1nc(c(s1)c1nsnc1)c1csc(n1)c1cccs1,1466.57
104,Oc1ccc2c(c1c1cocc1)c(ccc2)c1ccccn1,1207.67
394,Oc1ccc(s1)N1CNC(NC1)c1cnccn1,1355.15
377,n1csc(n1)C1(CCCC1)c1csc(c1)c1scnc1,1311.30
124,Oc1ccoc1c1ccnc(c1)c1ccc2c(c1)cccc2,1226.34
68,SC1CCC(C1c1cnccn1)C1CSCCS1,1238.45
450,o1ccc(c1)c1sc(cc1c1ccsc1)c1cccc2c1cccc2,1246.33


# Run autoML for a regression task

In [3]:
from chemml.autoML import ModelScreener
MS = ModelScreener(df, target="density_Kg/m3", featurization=True, smiles="smiles", 
                   screener_type="regressor", output_file="testing.txt")
scores = MS.screen_models(n_best=10, multi_core=False)

Converting SMILES to ChemML Molecule objects: 100%|██████████| 25/25 [00:00<00:00, 33.19it/s]


featurizing molecules in batches of 1 ...
25/25 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12s 12ms/step
Merging batch features ...    [DONE]


Calculating RDKit descriptors: 100%|██████████| 25/25 [00:00<00:00, 75.94it/s]


split done!
Single-core complete 


------------------------- Screening complete for feature set CoulombMatrix, time taken: 12.637 seconds -------------------------

split done!
Single-core complete 


------------------------- Screening complete for feature set morganfingerprints_radius3, time taken: 10.62 seconds -------------------------

split done!
Single-core complete 


------------------------- Screening complete for feature set MACCS_radius3, time taken: 7.414 seconds -------------------------

split done!
Single-core complete 


------------------------- Screening complete for feature set hashedtopologicaltorsion_radius3, time taken: 9.241 seconds -------------------------

split done!
Single-core complete 


------------------------- Screening complete for feature set rdkit_descriptors, time taken: 8.615 seconds -------------------------

split done!
Single-core complete 


------------------------- Screening complete for feature set mord_descriptors, time taken: 12.721 seco

In [6]:
scores

,ME,MAE,MSE,RMSE,MSLE,RMSLE,MAPE,MaxAPE,RMSPE,MPE,MaxAE,deltaMaxE,r_squared,std,time(seconds),Model,parameters,Feature
20,4.958658,9.202485,110.243049,10.499669,0.000071,0.008434,0.728465,1.316502,0.839578,0.408984,16.304223,22.669963,0.921597,37.498218,1.085185,Ridge,"{'alpha': 24.6, 'copy_X': True, 'fit_intercept...",rdkit_descriptors
21,-0.134804,10.849114,131.351219,11.460856,0.000082,0.009029,0.851001,1.268376,0.900780,-0.005417,16.071465,24.440602,0.906586,37.498218,1.485934,Lasso,"{'alpha': 0.10000000000000002, 'copy_X': True,...",rdkit_descriptors
22,6.004871,13.340453,240.236552,15.499566,0.000155,0.012449,1.054068,1.934614,1.236701,0.501836,23.959223,34.962595,0.829149,37.498218,1.669786,ElasticNet,"{'alpha': 0.005455594781168523, 'copy_X': True...",rdkit_descriptors
25,20.133130,20.133130,462.501526,21.505849,0.000299,0.017300,1.590453,2.487119,1.713243,1.590453,30.801725,16.614218,0.671079,37.498218,5.489565,Lasso,"{'alpha': 0.10000000000000002, 'copy_X': True,...",mord_descriptors
18,8.346180,18.830843,618.279153,24.865220,0.000397,0.019930,1.492118,3.217256,1.966877,0.652719,40.765533,54.517137,0.560293,37.498218,2.773437,Lasso,"{'alpha': 0.02335721469090124, 'copy_X': True,...",hashedtopologicaltorsion_radius3
24,18.924243,18.924243,750.990007,27.404197,0.000507,0.022512,1.519058,3.778707,2.211152,1.519058,46.797399,44.380714,0.465912,37.498218,2.807250,Ridge,"{'alpha': 0.1, 'copy_X': True, 'fit_intercept'...",mord_descriptors
26,22.331589,22.331589,757.486519,27.522473,0.000506,0.022492,1.781698,3.580846,2.213932,1.781698,44.346986,37.990156,0.461292,37.498218,6.315701,ElasticNet,"{'alpha': 0.06951927961775611, 'copy_X': True,...",mord_descriptors
17,28.962776,30.300355,1937.674264,44.019022,0.001334,0.036525,2.419969,6.049960,3.546499,2.314406,74.925732,76.932101,-0.378033,37.498218,2.593153,Ridge,"{'alpha': 5.0, 'copy_X': True, 'fit_intercept'...",hashedtopologicaltorsion_radius3
9,31.351781,34.734578,2105.141694,45.881823,0.001275,0.035702,2.663523,5.706163,3.477064,2.396549,75.797811,80.872007,-0.497132,37.498218,0.834708,SVR,"{'C': 79.1578947368421, 'cache_size': 200, 'co...",MACCS_radius3
23,29.543333,36.190000,2278.924900,47.738086,0.001377,0.037101,2.765244,6.017239,3.606768,2.228552,79.930000,89.900000,-0.620723,37.498218,2.737811,DecisionTreeRegressor,"{'ccp_alpha': 0.0, 'criterion': 'absolute_erro...",mord_descriptors


# Save scores to csv

In [5]:
scores.to_csv("autoML_test.csv",index=False)